## Project Overview

This project implements an **Equity Gap Analyzer** as a multi agent system that identifies and explains differences in student outcomes across subgroups using the public *StudentsPerformance* dataset.

The dataset includes student level test scores in math, reading, and writing, along with subgroup attributes such as gender, race_ethnicity, parental_level_of_education, lunch status, and test_preparation_course. The system focuses on questions of the form:

> "Analyze `<outcome>` equity gaps by `<subgroup>` and produce an interpretable equity report."

For example, the run below analyzes **reading_score** equity gaps by **race_ethnicity**.


## Agent Architecture

The system is implemented with the Google AI Developer Kit (ADK) and Gemini models. It uses four main agents:

1. **CoordinatorAgent**  
   - Entry point for user missions.  
   - Interprets the mission text, selects an outcome column and subgroup column, and orchestrates the other agents.  
   - Enforces the use of exact column names from the dataset schema.

2. **DataQualityAgent**  
   - Wraps the `profile_dataset` Python tool.  
   - Computes basic data quality diagnostics: missingness rates for the outcome and subgroup columns, subgroup counts, and flags for small groups below a minimum size threshold.  
   - Produces a structured dictionary summarizing data quality and caveats.

3. **AnalystAgent**  
   - Wraps the `compute_subgroup_stats` Python tool.  
   - Aggregates the outcome by subgroup, computing N, mean, and standard deviation per group.  
   - Computes pairwise gaps relative to a reference group (by default the largest group) and effect sizes based on pooled standard deviation.  
   - Returns a structured object with `subgroups`, `reference_group`, and `gaps`.

4. **WriterAgent**  
   - Invoked via an `AgentTool` as the final step in the pipeline.  
   - Takes the mission, data quality findings, and subgroup statistics as context and produces a concise, human readable equity report.  
   - The report includes:
     - a short context section,  
     - a markdown table of subgroup statistics,  
     - narrative interpretation of effect sizes (classified as small, moderate, large),  
     - concrete recommended next steps,  
     - and a data limitations section.

The agents are executed via an `InMemorySessionService` and `Runner`, which manage session state and event streaming for each run.


## Tools and State Management

Two custom Python tools are used:

- `profile_dataset(outcome_column, subgroup_column, min_group_size)`  
  - Uses pandas to compute missingness, subgroup counts, and a simple quality label.  
  - Runs deterministically against the global DataFrame.

- `compute_subgroup_stats(outcome_column, subgroup_column, reference_group=None)`  
  - Uses pandas and NumPy to compute per subgroup N, mean, standard deviation, differences vs reference, and pooled effect sizes.  
  - Returns a JSON compatible dictionary consumed by the WriterAgent.

Session management is implemented with `InMemorySessionService`. For each run:

1. A new session is created with the main DataFrame stored in session state.  
2. The `Runner` executes the CoordinatorAgent for that session, streaming events until a final response is produced.

In addition to ADK sessions, the notebook maintains a simple in memory `USER_MEMORY` object that tracks:

- the last preferred `outcome` column,  
- the last preferred `subgroup` column,  
- and a short run history.

A compact memory summary is prepended to each new mission so that, if a user later asks a vague question (for example "run the same analysis again but for writing"), the system has access to defaults reflecting previous choices.


## Logging and Observability

To support basic observability, the notebook maintains a `RUN_LOGS` list of dictionaries. Each completed run appends a row containing:

- `timestamp_utc` – the time of the run in UTC,  
- `mission` – the raw user mission text,  
- `report_preview` – the first 200 characters of the generated report,  
- `outcome_hint` – any inferred outcome column from the mission text,  
- `subgroup_hint` – any inferred subgroup column from the mission text.

For inspection in the notebook, `RUN_LOGS` is rendered as a pandas DataFrame. This provides a lightweight trace of how the system has been used over time, which missions were run, and which outcome/subgroup combinations were most common.


## Limitations and Future Work

There are several limitations in the current prototype:

- **Dataset constraints**  
  The public dataset anonymizes race and ethnicity into abstract labels such as Group A–E, which prevents the system from drawing conclusions about specific, named racial or ethnic communities. In a real deployment the same code could be pointed at de identified, but more richly labeled, internal datasets.

- **Statistical depth**  
  The current implementation focuses on mean differences and simple effect sizes. Future work could incorporate confidence intervals, regression based adjustment, or Bayesian models.

- **Model quota and evaluation**  
  Due to free tier rate limits for the Gemini API, the notebook only demonstrates a small number of full end to end runs. The evaluation of report quality is primarily qualitative and based on inspection, rather than using a separate LLM-as-judge loop.

Despite these constraints, the prototype exercises key agentic capabilities on an educational equity use case and provides a realistic template for extending to richer datasets and more advanced statistical methods.


In [1]:
# ============================================================
# Setup: imports and configuration
# ============================================================


import os
from kaggle_secrets import UserSecretsClient
import pandas as pd

from google import genai
from google.genai.types import HttpOptions

from google.adk.agents import LlmAgent
from google.adk.tools import ToolContext, AgentTool
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types

import asyncio
import uuid



try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Setup and authentication complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )


# Configure Gemini client
client = genai.Client(
    http_options=HttpOptions(api_version="v1")
)

MODEL_FAST = "gemini-2.5-flash"
MODEL_STRONG = "gemini-2.5-flash"   # for heavier reasoning, if desired

✅ Setup and authentication complete.


In [2]:
# ============================================================
# Load StudentsPerformance dataset
# ============================================================

DATA_PATH = "/kaggle/input/students-performance-in-exams/StudentsPerformance (1).csv"

df = pd.read_csv(DATA_PATH)

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
      .str.replace("/", "_")
)

df.head()



,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [3]:
# ============================================================
# Simple run logging / tracing
# ============================================================

from datetime import datetime

RUN_LOGS: list[dict] = []  # in-memory log of runs


def log_run(mission: str, report_text: str, extra: dict | None = None) -> None:
    """
    Append a simple log entry for each completed run.
    This gives you observability over missions, timestamps, and basic metadata.
    """
    entry = {
        "timestamp_utc": datetime.utcnow().isoformat(timespec="seconds"),
        "mission": mission,
        "report_preview": report_text[:200],
    }
    if extra:
        entry.update(extra)
    RUN_LOGS.append(entry)


def show_run_logs():
    """
    Display the logs as a DataFrame (if pandas is available),
    otherwise print them.
    """
    if not RUN_LOGS:
        print("No runs logged yet.")
        return

    try:
        import pandas as pd
        display(pd.DataFrame(RUN_LOGS))
    except Exception:
        for row in RUN_LOGS:
            print(row)


In [4]:
# ============================================================
# Simple user memory (preferences + history)
# ============================================================

KNOWN_OUTCOMES = ["math_score", "reading_score", "writing_score"]
KNOWN_SUBGROUPS = [
    "gender",
    "race_ethnicity",
    "parental_level_of_education",
    "lunch",
    "test_preparation_course",
]

USER_MEMORY = {
    "preferred_outcome": None,
    "preferred_subgroup": None,
    "run_history": [],  # list of dicts: {"outcome": ..., "subgroup": ...}
}


def infer_columns_from_mission(mission: str):
    """
    Heuristically infer outcome and subgroup columns mentioned in the mission
    by looking for known column names.
    """
    mission_lower = mission.lower()
    outcome = next((c for c in KNOWN_OUTCOMES if c in mission_lower), None)
    subgroup = next((c for c in KNOWN_SUBGROUPS if c in mission_lower), None)
    return outcome, subgroup


def build_memory_summary() -> str:
    """
    Build a compact text summary of user memory to feed into the coordinator context.
    """
    pref_outcome = USER_MEMORY["preferred_outcome"] or "None"
    pref_subgroup = USER_MEMORY["preferred_subgroup"] or "None"
    return (
        f"User memory summary: preferred_outcome={pref_outcome}, "
        f"preferred_subgroup={pref_subgroup}. "
        f"Use these as defaults if the mission is ambiguous."
    )


In [5]:
# ============================================================
# Data quality profiling tool (simplified for ADK auto schema)
# ============================================================

def profile_dataset(
    outcome_column: str,
    subgroup_column: str,
    min_group_size: int,
) -> dict:
    """
    Profiles the dataset for basic data quality issues that affect equity analysis.

    Args:
        outcome_column:
            Name of the numeric outcome column to analyze,
            for example 'math_score'.
        subgroup_column:
            Name of the categorical subgroup field,
            for example 'gender' or 'race_ethnicity'.
        min_group_size:
            Minimum acceptable group size for reporting,
            for example 10.

    Returns:
        A dictionary with:
          - 'missing_stats': per column missingness (percent)
          - 'subgroup_counts': subgroup value counts
          - 'small_groups': list of groups below min_group_size
          - 'quality_summary': short machine readable judgment
    """
    global df  # use the globally loaded dataframe

    if outcome_column not in df.columns:
        raise ValueError(f"Outcome column '{outcome_column}' not found in dataframe.")
    if subgroup_column not in df.columns:
        raise ValueError(f"Subgroup column '{subgroup_column}' not found in dataframe.")

    # Basic missingness information (%)
    missing = df[[outcome_column, subgroup_column]].isna().mean().mul(100).to_dict()

    # Subgroup counts
    subgroup_counts = (
        df[subgroup_column]
        .value_counts(dropna=False)
        .to_dict()
    )

    # Small groups
    small_groups = [
        {"group_value": str(val), "count": int(cnt)}
        for val, cnt in subgroup_counts.items()
        if cnt < min_group_size
    ]

    # Simple label for quality
    if missing[outcome_column] > 10:
        quality = "high_missingness_outcome"
    elif len(small_groups) > 0:
        quality = "small_subgroups_present"
    else:
        quality = "acceptable"

    return {
        "missing_stats": missing,
        "subgroup_counts": subgroup_counts,
        "small_groups": small_groups,
        "quality_summary": quality,
    }


In [6]:
# ============================================================
# Subgroup statistics and gap calculation tool
# ============================================================

from typing import Optional
import numpy as np

def compute_subgroup_stats(
    outcome_column: str,
    subgroup_column: str,
    reference_group: Optional[str] = None,
) -> dict:
    """
    Computes subgroup level descriptive statistics and equity gaps
    for the specified outcome and subgroup dimension.

    Args:
        outcome_column:
            Name of the numeric outcome column.
        subgroup_column:
            Name of the categorical field defining the subgroups.
        reference_group:
            Optional reference subgroup value to compare against.
            If None, chooses the largest subgroup as reference.

    Returns:
        A dictionary with:
          - 'outcome'
          - 'subgroup_field'
          - 'subgroups': list of {value, n, mean, std}
          - 'reference_group'
          - 'gaps': list of comparisons with difference and effect size
    """
    global df

    if outcome_column not in df.columns:
        raise ValueError(f"Outcome column '{outcome_column}' not found.")
    if subgroup_column not in df.columns:
        raise ValueError(f"Subgroup column '{subgroup_column}' not found.")

    working = df[[outcome_column, subgroup_column]].dropna()

    # Aggregate descriptive statistics by subgroup
    grouped = (
        working
        .groupby(subgroup_column, dropna=False)[outcome_column]
        .agg(["count", "mean", "std"])
        .reset_index()
        .rename(
            columns={
                subgroup_column: "value",
                "count": "n",
                "mean": "mean",
                "std": "std",
            }
        )
    )

    grouped["std"] = grouped["std"].fillna(0.0)

    subgroups = grouped.to_dict(orient="records")

    # Choose reference group
    if reference_group is None:
        ref_row = grouped.sort_values("n", ascending=False).iloc[0]
    else:
        mask = grouped["value"].astype(str) == str(reference_group)
        if not mask.any():
            raise ValueError(
                f"Reference group '{reference_group}' not present in '{subgroup_column}'."
            )
        ref_row = grouped[mask].iloc[0]

    ref_value = str(ref_row["value"])
    ref_mean = float(ref_row["mean"])
    ref_std = float(ref_row["std"])

    gaps = []
    for _, row in grouped.iterrows():
        comp_value = str(row["value"])
        if comp_value == ref_value:
            continue

        diff = float(row["mean"] - ref_mean)

        n1 = int(ref_row["n"])
        n2 = int(row["n"])
        pooled_var = (
            ((n1 - 1) * ref_std**2 + (n2 - 1) * float(row["std"])**2)
            / max(n1 + n2 - 2, 1)
        )
        pooled_sd = float(np.sqrt(max(pooled_var, 1e-8)))
        effect_size = diff / pooled_sd if pooled_sd > 0 else 0.0

        gaps.append(
            {
                "reference": ref_value,
                "comparison": comp_value,
                "difference": diff,
                "effect_size": effect_size,
                "n_reference": n1,
                "n_comparison": n2,
            }
        )

    return {
        "outcome": outcome_column,
        "subgroup_field": subgroup_column,
        "subgroups": subgroups,
        "reference_group": ref_value,
        "gaps": gaps,
    }


In [7]:
# ============================================================
# Research insights tool (simple Gemini-backed)
# ============================================================

def retrieve_research_insights(
    subgroup_type: str,
    outcome_type: str,
) -> dict:
    """
    Uses Gemini to generate short, research-informed insights
    about typical interventions for the given subgroup and outcome.

    Args:
        subgroup_type:
            A short label such as 'students_with_disabilities',
            'english_learners', 'gender', or 'race_ethnicity'.
        outcome_type:
            A label such as 'math_achievement', 'reading_achievement',
            'overall_academic_performance'.

    Returns:
        A dictionary with:
          - 'raw_text': a short narrative with bullet points and summary.
    """

    prompt = f"""
    You are an education researcher. Summarize three evidence-informed strategies
    that have been shown to help reduce inequities in {outcome_type}
    for the subgroup type {subgroup_type}.
    Focus on interventions that a K–12 school could realistically implement.
    Provide three concise bullet points and then a short paragraph summary.
    """

    response = client.models.generate_content(
        model=MODEL_STRONG,
        contents=prompt,
    )

    text = "".join(
        getattr(part, "text", "")
        for part in response.candidates[0].content.parts
    )

    return {
        "raw_text": text
    }


In [8]:
# ============================================================
# DataQualityAgent
# ============================================================

data_quality_agent = LlmAgent(
    model=MODEL_FAST,
    name="data_quality_agent",
    description="Profiles the dataset and identifies issues that affect equity analysis.",
    instruction=(
        "You are a data quality analyst for K–12 assessment data. "
        "Given the available tools and the user mission, call the profiling tool "
        "exactly once with appropriate parameters. "
        "Then summarize the main issues in a short JSON friendly explanation."
    ),
    tools=[profile_dataset],
)


In [9]:
# ============================================================
# AnalystAgent
# ============================================================

analyst_agent = LlmAgent(
    model=MODEL_FAST,
    name="analyst_agent",
    description=(
        "Computes subgroup level statistics and equity gaps "
        "for the selected outcome and subgroup dimension."
    ),
    instruction=(
        "You are a quantitative analyst. "
        "Use the subgroup statistics tool to compute group means and gaps. "
        "Pick a sensible reference group if the user does not specify one, "
        "often the group with the largest N. "
        "Return only a concise textual explanation plus the raw tool result."
    ),
    tools=[compute_subgroup_stats],
)


In [10]:
# ============================================================
# InsightsAgent
# ============================================================

insights_agent = LlmAgent(
    model=MODEL_STRONG,
    name="insights_agent",
    description=(
        "Interprets equity gaps and links them to research informed strategies."
    ),
    instruction=(
        "You are an education equity expert. "
        "Given subgroup gap data and a general sense of the subgroup type "
        "and outcome type, use the research insights tool to obtain background. "
        "Then produce a short list of key interpretive points."
    ),
    tools=[retrieve_research_insights],
)


In [11]:
# ============================================================
# WriterAgent (updated for table + effect size interpretation)
# ============================================================

writer_agent = LlmAgent(
    model=MODEL_STRONG,
    name="writer_agent",
    description="Writes a short equity gap report suitable for school leaders.",
    instruction=(
        "You are a precise but accessible education data writer.\n\n"
        "You receive, in the conversation history, outputs from tools including:\n"
        "  - Data quality results with fields like 'missing_stats', 'small_groups', and 'quality_summary'.\n"
        "  - Subgroup statistics and gaps with fields:\n"
        "        outcome, subgroup_field,\n"
        "        subgroups: list of {value, n, mean, std},\n"
        "        reference_group,\n"
        "        gaps: list of {reference, comparison, difference, effect_size, n_reference, n_comparison}.\n"
        "  - Research insights with a 'raw_text' field that contains bullet points and a summary paragraph.\n\n"
        "Write a concise report with these sections:\n"
        "1. Context and outcome: Briefly state the outcome (for example reading_score) and subgroup dimension.\n"
        "2. Key equity findings: First render a markdown table of subgroup statistics with columns:\n"
        "       Group, N, Mean Score, Difference vs reference, Effect size\n"
        "   using the 'subgroups' and 'gaps' information. The reference group's difference is '0' and effect size '0.00'.\n"
        "   Below the table, summarize which groups are lower or higher than the reference.\n"
        "3. Interpretation: Explain the size of gaps in plain language.\n"
        "   When effect sizes are available, classify them as 'small' (~0.2), 'moderate' (~0.5), or 'large' (~0.8 or more),\n"
        "   and mention this classification explicitly.\n"
        "4. Suggested next steps: Provide 2–4 concrete, actionable strategies drawn from the research insights text.\n"
        "5. Data limitations: Note missingness, small-group caveats, and the fact that race_ethnicity groups in this\n"
        "   public dataset are anonymized as labels like Group A–E, not real named racial groups.\n\n"
        "Style requirements:\n"
        " - Use clear headings starting with '**1. Context and Outcome**', etc.\n"
        " - Keep the full report under about 700 words.\n"
        " - Write for non-technical school leaders: formal but plain language.\n"
        " - Do not show raw JSON; instead, convert the tool outputs into readable prose and tables.\n"
    ),
)

writer_tool = AgentTool(agent=writer_agent)


In [12]:
# ============================================================
# CoordinatorAgent
# ============================================================

coordinator_agent = LlmAgent(
    model=MODEL_STRONG,
    name="coordinator_agent",
    description=(
        "Coordinates data quality profiling, gap analysis, "
        "insight generation, and report writing for an educational equity analysis system."
    ),
    instruction=(
        "You are the Coordinator for the Equity Gap Analyzer. "
        "You orchestrate calls to the following specialist agents: "
        " - data_quality_agent "
        " - analyst_agent "
        " - insights_agent "
        " - writer_agent (via writer_tool). "

        "The dataset has exactly these columns: "
        "  gender, race_ethnicity, parental_level_of_education, lunch, "
        "  test_preparation_course, math_score, reading_score, writing_score. "

        "VALID outcome columns: "
        "  math_score, reading_score, writing_score. "

        "VALID subgroup columns include: "
        "  gender, race_ethnicity, parental_level_of_education, lunch, "
        "  test_preparation_course. "

        "When the user describes an outcome or subgroup, map it to the exact "
        "column name above. NEVER invent new column names. Never use variations "
        "like 'reading score' or 'race/ethnicity'. Only use the exact names: "
        "'reading_score' and 'race_ethnicity'. "

        "PROCESS: "
        "1. Identify the correct outcome_column and subgroup_column from the user mission. "
        "2. Call data_quality_agent using the tool call: "
        "       profile_dataset(outcome_column=..., subgroup_column=..., min_group_size=10). "
        "3. Call analyst_agent using: "
        "       compute_subgroup_stats(outcome_column=..., subgroup_column=...). "
        "4. Call insights_agent using: "
        "       retrieve_research_insights(subgroup_type=..., outcome_type=...). "
        "   - subgroup_type should map to the subgroup such as 'gender' or 'race_ethnicity'. "
        "   - outcome_type should be a phrase like 'reading achievement', "
        "     'math achievement', or 'writing achievement'. "
        "5. Call writer_agent using writer_tool, passing: "
        "       - the user mission "
        "       - quality findings "
        "       - subgroup statistics and gaps "
        "       - insights text "
        "6. Return ONLY the writer_agent's final written report. "

        "IMPORTANT: "
        " - Use only one outcome column and one subgroup column per mission. "
        " - Use EXACT dataframe column names for tool calls. "
        " - Never use statistical terminology unless included by the other agents. "
        " - Your job is orchestration, not analysis. Let the tools produce the numbers. "
    ),
    tools=[
        AgentTool(agent=data_quality_agent),
        AgentTool(agent=analyst_agent),
        AgentTool(agent=insights_agent),
        writer_tool,
    ],
)



In [13]:
# === App constants, session service, and runner ===

APP_NAME = "equity_gap_analyzer"
USER_ID = "demo_user"  # can be made dynamic later

# Single in-memory session service for this notebook
session_service = InMemorySessionService()

# Runner executes the top-level coordinator agent
runner = Runner(
    agent=coordinator_agent,
    app_name=APP_NAME,
    session_service=session_service,
)



In [14]:
# === Async session creation helper ===

async def create_equity_session_async(df: pd.DataFrame):
    """
    Create a fresh ADK session with the dataframe stored in session state.

    Returns:
        A Session object whose `state["df"]` contains the analysis dataframe.
    """
    session_id = f"equity_session_{uuid.uuid4().hex[:8]}"

    session = await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        state={"df": df},
        session_id=session_id,
    )
    return session


In [15]:
# ============================================================
# Core async entrypoint (with memory + logging)
# ============================================================

async def run_equity_gap_analysis(
    mission: str = (
        "Analyze math_score equity gaps by gender. "
        "Assume 'math_score' is the outcome column and 'gender' is the subgroup column."
    )
) -> str:
    """
    Asynchronous routine that:
      1. Creates a new session with df in its state.
      2. Sends a memory-augmented mission to the coordinator_agent through the Runner.
      3. Streams events and returns the final response text.
      4. Updates USER_MEMORY and RUN_LOGS.
    """
    # 1. Create a new session and attach df to its state
    session = await create_equity_session_async(df)

    # 2. Infer outcome/subgroup hints from the mission
    inferred_outcome, inferred_subgroup = infer_columns_from_mission(mission)

    # 3. Build a compact memory summary to prepend to the mission
    memory_summary = build_memory_summary()
    full_user_message = (
        memory_summary
        + "\n\nCurrent mission:\n"
        + mission
    )

    content = types.Content(
        role="user",
        parts=[types.Part(text=full_user_message)],
    )

    # 4. Stream events from the Runner and capture the final response text
    final_text = None

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=session.id,
        new_message=content,
    ):
        evt_content = getattr(event, "content", None)
        if evt_content and getattr(evt_content, "parts", None):
            for part in evt_content.parts:
                if hasattr(part, "text") and part.text:
                    final_text = part.text

    if final_text is None:
        final_text = "No response text was generated by the runner."

    # 5. Update simple memory (if we learned anything new from the mission)
    if inferred_outcome:
        USER_MEMORY["preferred_outcome"] = inferred_outcome
    if inferred_subgroup:
        USER_MEMORY["preferred_subgroup"] = inferred_subgroup

    USER_MEMORY["run_history"].append(
        {
            "mission": mission,
            "outcome": inferred_outcome,
            "subgroup": inferred_subgroup,
        }
    )

    # 6. Log this run for observability
    log_run(
        mission=mission,
        report_text=final_text,
        extra={
            "outcome_hint": inferred_outcome,
            "subgroup_hint": inferred_subgroup,
        },
    )

    return final_text


## Uncomment out the following cell when ready to run

In [16]:
#report_text = await run_equity_gap_analysis(
#    "Analyze reading_score equity gaps by race_ethnicity."
#)
#print(report_text)
#
#show_run_logs()  
#USER_MEMORY    
